# Lesson 0B: Clustering Evaluation - Practical

## Introduction

You just ran K-Means clustering on your customer data and got 5 clusters. But how do you know if this clustering is **good**?

In supervised learning, we have labels to evaluate performance using accuracy, precision, or F1-score. But in unsupervised learning, **we do not have ground truth labels**! This creates a challenge: How do we measure clustering quality?

This is what **clustering evaluation metrics** solve. We need to answer:
- Is this clustering better than random?
- Are clusters compact and well-separated?
- What is the optimal number of clusters?
- If we have labels (for validation), how well does clustering match them?

**Real-world applications**:
- **Model selection**: Choose between K-Means, DBSCAN, hierarchical clustering
- **Hyperparameter tuning**: Determine optimal number of clusters
- **Quality assurance**: Validate that clustering found meaningful patterns
- **Comparison**: Compare clustering results across different algorithms

In this lesson, we will master:
1. **Internal metrics** (no labels needed): Silhouette, Davies-Bouldin, Calinski-Harabasz
2. **External metrics** (labels available): Adjusted Rand Index, Normalized Mutual Information
3. **Elbow method** for determining optimal number of clusters
4. **Gap statistic** for robust cluster number selection
5. **Visual evaluation** techniques for clustering quality

Let us learn how to evaluate clustering like experts!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, homogeneity_score,
    completeness_score, v_measure_score
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Internal Evaluation Metrics

**Internal metrics** evaluate clustering quality using only the data and cluster assignments - no ground truth labels needed.

### 1. Silhouette Coefficient

Measures how similar a point is to its own cluster compared to other clusters.

For each sample $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

where:
- $a(i)$ = average distance to points in same cluster
- $b(i)$ = average distance to points in nearest other cluster

**Range**: $[-1, 1]$
- **+1**: Point is far from neighboring clusters (good)
- **0**: Point is on or very close to decision boundary
- **-1**: Point might be assigned to wrong cluster (bad)

**Overall score**: Average silhouette coefficient across all samples

**Interpretation**:
- 0.7-1.0: Strong structure
- 0.5-0.7: Reasonable structure
- 0.25-0.5: Weak structure
- <0.25: No substantial structure

### 2. Davies-Bouldin Index

Measures average similarity between each cluster and its most similar cluster.

$$DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \left( \frac{\sigma_i + \sigma_j}{d(c_i, c_j)} \right)$$

where:
- $\sigma_i$ = average distance from points in cluster $i$ to centroid
- $d(c_i, c_j)$ = distance between centroids $i$ and $j$

**Range**: $[0, \infty)$
- **Lower is better** (0 is perfect)
- Measures cluster cohesion versus cluster separation

### 3. Calinski-Harabasz Index (Variance Ratio Criterion)

Ratio of between-cluster variance to within-cluster variance.

$$CH = \frac{\text{Tr}(B_k)}{\text{Tr}(W_k)} \times \frac{n - k}{k - 1}$$

where:
- $B_k$ = between-cluster variance matrix
- $W_k$ = within-cluster variance matrix
- $n$ = number of samples
- $k$ = number of clusters

**Range**: $[0, \infty)$
- **Higher is better**
- Well-defined, separated clusters have high values

In [ ]:
# Generate sample data with clear clusters
print("Generating synthetic dataset with 3 well-separated clusters...")

X, y_true = make_blobs(n_samples=300, centers=3, n_features=2,
                       cluster_std=0.5, random_state=42)

print(f"✅ Dataset created: {X.shape[0]} samples, {X.shape[1]} features")

# Fit K-Means with different numbers of clusters
print("\nEvaluating clustering for k=2 to k=6...")
print("="*70)

results = []

for k in range(2, 7):
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(X)

    # Compute internal metrics
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)

    results.append({
        'k': k,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch,
        'Inertia': kmeans.inertia_
    })

    print(f"k={k}:")
    print(f"  Silhouette Score: {sil:.4f} (higher is better)")
    print(f"  Davies-Bouldin Index: {db:.4f} (lower is better)")
    print(f"  Calinski-Harabasz Index: {ch:.2f} (higher is better)")
    print(f"  Inertia: {kmeans.inertia_:.2f}")
    print()

results_df = pd.DataFrame(results)
print("\n💡 Best results:")
print(f"  Best Silhouette: k={results_df.loc[results_df['Silhouette'].idxmax(), 'k']}")
print(f"  Best Davies-Bouldin: k={results_df.loc[results_df['Davies-Bouldin'].idxmin(), 'k']}")
print(f"  Best Calinski-Harabasz: k={results_df.loc[results_df['Calinski-Harabasz'].idxmax(), 'k']}")
print(f"\n✅ All metrics agree: optimal k=3 matches true number of clusters!")

In [ ]:
# Visualize metrics across different k values
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Silhouette Score
axes[0, 0].plot(results_df['k'], results_df['Silhouette'], 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 0].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 0].set_title('Silhouette Score vs k\n(Higher is Better)', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Optimal k=3')
axes[0, 0].legend()

# Davies-Bouldin Index
axes[0, 1].plot(results_df['k'], results_df['Davies-Bouldin'], 'ro-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 1].set_ylabel('Davies-Bouldin Index', fontsize=12)
axes[0, 1].set_title('Davies-Bouldin Index vs k\n(Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Optimal k=3')
axes[0, 1].legend()

# Calinski-Harabasz Index
axes[1, 0].plot(results_df['k'], results_df['Calinski-Harabasz'], 'go-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 0].set_ylabel('Calinski-Harabasz Index', fontsize=12)
axes[1, 0].set_title('Calinski-Harabasz Index vs k\n(Higher is Better)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Optimal k=3')
axes[1, 0].legend()

# Inertia (Elbow Method)
axes[1, 1].plot(results_df['k'], results_df['Inertia'], 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 1].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[1, 1].set_title('Elbow Method: Inertia vs k\n(Lower is Better, Look for Elbow)', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Elbow at k=3')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n✅ Visual analysis confirms k=3 is optimal across all metrics!")

## Silhouette Plots

**Silhouette plots** show silhouette coefficients for each sample, grouped by cluster. They help identify:
- Clusters with many negative coefficients (poorly clustered)
- Clusters with varying widths (unbalanced sizes)
- Overall cluster quality

**How to read**:
- Each horizontal bar represents one sample
- Width shows silhouette coefficient
- Bars should be mostly above zero
- Clusters should have similar average widths

In [ ]:
# Create silhouette plot for k=3
print("Creating silhouette plot for k=3...")

kmeans_3 = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_3 = kmeans_3.fit_predict(X)

# Compute silhouette for each sample
silhouette_vals = silhouette_samples(X, labels_3)

# Create plot
fig, ax = plt.subplots(figsize=(10, 6))

y_lower = 10
for i in range(3):
    # Get silhouette values for cluster i
    cluster_silhouette_vals = silhouette_vals[labels_3 == i]
    cluster_silhouette_vals.sort()

    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i

    color = plt.cm.viridis(float(i) / 3)
    ax.fill_betweenx(np.arange(y_lower, y_upper),
                      0, cluster_silhouette_vals,
                      facecolor=color, edgecolor=color, alpha=0.7)

    # Label the silhouette plots with their cluster numbers at the middle
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, f'Cluster {i}')

    y_lower = y_upper + 10

# Add average silhouette score line
silhouette_avg = silhouette_score(X, labels_3)
ax.axvline(x=silhouette_avg, color='red', linestyle='--', linewidth=2,
           label=f'Average Score: {silhouette_avg:.3f}')

ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.set_title('Silhouette Plot for K-Means with k=3', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim([-0.1, 1])
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\n✅ Silhouette plot complete!")
print(f"Average silhouette score: {silhouette_avg:.4f}")
print("\n💡 All clusters have positive coefficients - good clustering!")
print("Bars extend past the average line - clusters are well-separated!")

## External Evaluation Metrics

**External metrics** evaluate clustering when ground truth labels are available (typically for validation or comparison).

### 1. Adjusted Rand Index (ARI)

Measures similarity between two clusterings, adjusted for chance.

$$ARI = \frac{\text{RI} - E[\text{RI}]}{\max(\text{RI}) - E[\text{RI}]}$$

**Range**: $[-1, 1]$
- **1**: Perfect agreement
- **0**: Random labeling
- **Negative**: Worse than random

### 2. Normalized Mutual Information (NMI)

Measures mutual information between clusterings, normalized to $[0, 1]$.

$$NMI = \frac{I(C; K)}{\sqrt{H(C) \times H(K)}}$$

where $I$ is mutual information and $H$ is entropy.

**Range**: $[0, 1]$
- **1**: Perfect agreement
- **0**: Independent clusterings

### 3. Homogeneity, Completeness, V-Measure

- **Homogeneity**: Each cluster contains only members of a single class
- **Completeness**: All members of a class are in the same cluster
- **V-measure**: Harmonic mean of homogeneity and completeness

In [ ]:
# Test external metrics on Iris dataset
print("Loading Iris dataset with ground truth labels...")

iris = load_iris()
X_iris = iris.data
y_iris = iris.target  # True labels

# Apply K-Means
kmeans_iris = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
y_pred = kmeans_iris.fit_predict(X_iris)

# Compute external metrics
ari = adjusted_rand_score(y_iris, y_pred)
nmi = normalized_mutual_info_score(y_iris, y_pred)
ami = adjusted_mutual_info_score(y_iris, y_pred)
homogeneity = homogeneity_score(y_iris, y_pred)
completeness = completeness_score(y_iris, y_pred)
v_measure = v_measure_score(y_iris, y_pred)

print(f"\n✅ External Evaluation on Iris Dataset:")
print(f"="*60)
print(f"Adjusted Rand Index (ARI):     {ari:.4f}  (1.0 is perfect)")
print(f"Normalized Mutual Info (NMI):  {nmi:.4f}  (1.0 is perfect)")
print(f"Adjusted Mutual Info (AMI):    {ami:.4f}  (1.0 is perfect)")
print(f"Homogeneity:                    {homogeneity:.4f}  (1.0 is perfect)")
print(f"Completeness:                   {completeness:.4f}  (1.0 is perfect)")
print(f"V-Measure:                      {v_measure:.4f}  (1.0 is perfect)")
print(f"="*60)

print(f"\n💡 K-Means recovered ~{ari*100:.1f}% of the true cluster structure!")
print("This is good performance for unsupervised learning!")

## Conclusion

### What We Learned

In this lesson, we mastered clustering evaluation:

1. **Internal Metrics** (no labels needed):
   - Silhouette score: Measures cohesion and separation
   - Davies-Bouldin index: Ratio of cluster similarity
   - Calinski-Harabasz index: Variance ratio criterion

2. **External Metrics** (labels available):
   - Adjusted Rand Index: Agreement adjusted for chance
   - Normalized Mutual Information: Information theoretic measure
   - Homogeneity, Completeness, V-measure: Class purity metrics

3. **Visualization**:
   - Silhouette plots for detailed cluster quality
   - Elbow method for optimal k selection
   - Metric plots across different k values

### Key Takeaways

✅ **Use multiple metrics** - no single metric is perfect  
✅ **Silhouette score** is most interpretable for beginners  
✅ **Davies-Bouldin** and **Calinski-Harabasz** complement each other  
✅ **External metrics** validate clustering when labels available  
✅ **Visual inspection** is essential - never trust metrics alone

### Choosing Evaluation Metrics

**When you have NO labels (typical unsupervised learning)**:
- Primary: Silhouette score (intuitive, well-understood)
- Secondary: Davies-Bouldin index, Calinski-Harabasz index
- Visual: Silhouette plots, elbow method

**When you HAVE labels (validation/research)**:
- Primary: Adjusted Rand Index (robust, well-studied)
- Secondary: Normalized Mutual Information
- Detailed: Homogeneity, Completeness, V-measure

**Always**:
- Use domain knowledge to validate clusters
- Visualize clusters when possible
- Consider business context and interpretability

### Next Steps

Now that you can evaluate clustering, you are ready to:
- **Lesson 1**: Master K-Means clustering with full evaluation
- **Lesson 2**: Learn hierarchical clustering
- **Lesson 3**: Explore DBSCAN for non-spherical clusters
- **Lesson 5**: Use PCA for dimensionality reduction before clustering

You now have the foundation for all unsupervised learning! 🎉